In [ ]:
# If needed

# import os
# os.chdir(".../atmosphere-profile-retrieval-dense-nn")
# os.getcwd()

In [ ]:
import torch
from torch.utils.data import TensorDataset, DataLoader

import numpy as np
%matplotlib widget
import matplotlib.pyplot as plt

import humanize as h

import training.models as models
from paths import H_GRID_PATH, CHECKPOINT_DIR, AH_INPUTS_TRAIN_PATH, BT_INPUTS_TRAIN_PATH, CZ_INPUTS_TRAIN_PATH, DATE_INPUTS_TRAIN_PATH, GHEIGHT_INPUTS_TRAIN_PATH, GAH_INPUTS_TRAIN_PATH, MM_INPUTS_TRAIN_PATH, AH_INPUTS_TEST_PATH, BT_INPUTS_TEST_PATH, CZ_INPUTS_TEST_PATH, DATE_INPUTS_TEST_PATH, GHEIGHT_INPUTS_TEST_PATH, GAH_INPUTS_TEST_PATH, MM_INPUTS_TEST_PATH


def model_stats(model):
    total_params = sum(p.numel() for p in model.parameters())
    param_bytes = sum(p.numel() * p.element_size() for p in model.parameters())
    
    print(f"Model size: {h.intword(total_params, format='%.2f')} parameters ({h.naturalsize(param_bytes, binary=True, format='%.2f')})\n")

In [ ]:
def generate_loaders(X_names, batch_size):
    if "AH" in X_names and "BT" in X_names:
        raise Exception("Model has to use either BT or AH as main input, but not both.")

    if "AH" not in X_names and "BT" not in X_names:
        raise Exception("Model has to use either BT or AH as main input, but not both.")
    
    name_paths_dict = {
        "BT": (BT_INPUTS_TRAIN_PATH, BT_INPUTS_TEST_PATH),
        "CZ": (CZ_INPUTS_TRAIN_PATH, CZ_INPUTS_TEST_PATH),
        "DATE": (DATE_INPUTS_TRAIN_PATH, DATE_INPUTS_TEST_PATH),
        "GH": (GHEIGHT_INPUTS_TRAIN_PATH, GHEIGHT_INPUTS_TEST_PATH),
        "GAH": (GAH_INPUTS_TRAIN_PATH, GAH_INPUTS_TEST_PATH),
        "MM": (MM_INPUTS_TRAIN_PATH, MM_INPUTS_TEST_PATH),
        "AH": (AH_INPUTS_TRAIN_PATH, AH_INPUTS_TEST_PATH)
    }
    
    X_list_train = []
    X_list_test = []
    for name in X_names:
        train_path, test_path = name_paths_dict[name]

        train_arr = np.load(train_path)
        test_arr = np.load(test_path)

        if name == "AH":
            train_arr = np.nan_to_num(train_arr, nan=0.0)
            test_arr = np.nan_to_num(test_arr, nan=0.0)

        X_list_train.append(train_arr)
        X_list_test.append(test_arr)

    
    X_train = np.hstack(X_list_train)
    X_test = np.hstack(X_list_test)

    if "BT" in X_names:
        Y_train = np.load(AH_INPUTS_TRAIN_PATH)
        Y_test = np.load(AH_INPUTS_TEST_PATH)
    else:
        Y_train = np.load(BT_INPUTS_TRAIN_PATH)
        Y_test = np.load(BT_INPUTS_TEST_PATH)

    train_dataset = TensorDataset(torch.from_numpy(X_train), torch.from_numpy(Y_train))
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    
    test_dataset = TensorDataset(torch.from_numpy(X_test), torch.from_numpy(Y_test))
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=True)

    return train_loader, test_loader, X_train.shape[1], Y_train.shape[1]


def masked_mse_loss(Y_pred, Y_true):
    mask = ~torch.isnan(Y_true)
    diff = Y_pred[mask] - Y_true[mask]
    
    return torch.mean(diff ** 2)

In [ ]:
def rand_profile(model, loader, loss):
    model_name = f"{'-'.join(X_names)}.{type(model).__name__}"
    checkpoint_path = CHECKPOINT_DIR / f"{model_name}.pt"
    
    model.load_state_dict(torch.load(checkpoint_path, map_location="cpu"))
    model.eval()

    h_grid = np.load(H_GRID_PATH)

    rng = np.random.default_rng()

    with torch.no_grad():
        for X, Y in loader:
            rand_ind = rng.choice(X.size(0), size=4, replace=False)
            
            x = X[rand_ind]
            y_true = Y[rand_ind]

            y_pred = model(x)

            # График
            plt.close('all')
            fig, ax = plt.subplots(1, 4, figsize=(11, 5), sharey=True, layout='constrained')

            plt.suptitle(model_name)

            for a in ax:
                a.grid(True, axis="y", linestyle="--", alpha=0.4)

            for i in range(4):
                y_true_cur = y_true[i]
                y_pred_cur = y_pred[i]
                
                valid = ~torch.isnan(y_true_cur)
                
                ax[i].plot(y_true_cur[valid], h_grid[valid], color="green", label="Interpolated")
                ax[i].plot(y_pred_cur[valid], h_grid[valid], color="black", linestyle="--", label="Predicted")
                
                ax[i].set_xlim(left=0)
                ax[i].set_xlabel(r'Absolute humidity, g/m$^3$')
                ax[i].set_title(f"Loss: {loss(y_pred_cur, y_true_cur):.3f}")

                ax[i].legend()

            ax[0].set_ylabel("Height, m")
            
        
            plt.show()

            break

In [ ]:
X_names = ["BT"]
batch_size = 32
loss = masked_mse_loss

train_loader, test_loader, features, targets = generate_loaders(X_names, batch_size)

In [ ]:
model = models.expanding_12(features, targets)
model_stats(model)

In [ ]:
rand_profile(model, test_loader, loss)